# G3 geometry on **Gemma-3-27B** (RunPod): premise check + four-cell screen

Two geometry analyses on the **target model at the true operating point** (`google/gemma-3-27b-it`, inject **L37/62**), from one pod run:

1. **G3 premise check** - per-arm aggregate `cos(v_concept, d_refusal)` across all layers, both pooling protocols, with the decision verdict (*harmful > benign, stable across protocols?*). This is the pending "funded-stage" 27B row of the findings log.
2. **Four-cell screen** - per-concept `cos` at L37 sorting concepts into the harm x refusal-geometry 2x2 (which concepts populate the scarce off-diagonal cells).

Both ride on the **same** `d_refusal` and concept vectors, so they cost one build. The Colab small-model run was feasibility; **these are the numbers you select concepts from.**

> 🔒 Inference-only on the **stock** model - no abliteration, no judge, no generation, no public port. Outputs are `cos` scalars + per-arm aggregates + word lists (safe to keep/download). Never persist vectors/activations; keep per-concept detail out of the public tree. Delete the pod when done - **regenerate, never archive** the weights. (CLAUDE.md)

## 0. Pod prerequisites (set these when creating the pod)

- **GPU:** 1x A100 80GB (value) or H100 80GB. A 48GB/40GB card cannot hold bf16 27B (~54GB).
- **Template:** RunPod **PyTorch 2.x** (2.8.0 is fine; pick a CUDA 12.x build). Jupyter preinstalled.
- **Container disk:** ~30GB (OS + pip; ephemeral).
- **Persistent volume on `/workspace`:** **>= 90GB** (the model cache alone is ~54GB).
- **Env var:** set **`HF_TOKEN`** to your Hugging Face token (accept the Gemma license on the gemma-3-27b-it page under that account).
- Add your **SSH public key** in RunPod Settings if you want terminal/`runpodctl` access (see chat).
- Open Jupyter, upload this notebook, **Run All**.

In [ ]:
# === cell 1: environment ===
import os, sys, subprocess, torch
os.environ.setdefault('HF_HOME', '/workspace/hf-cache')   # keep the 54GB cache on the big volume, off the container disk
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

assert torch.cuda.is_available(), 'No GPU visible - pick a GPU pod'
_vram = torch.cuda.get_device_properties(0).total_memory/1e9
print(f'GPU: {torch.cuda.get_device_name(0)}  |  VRAM {_vram:.0f} GB')
if _vram < 78:
    print('  !! WARNING: <80GB VRAM. bf16 gemma-3-27b is ~54GB weights; expect OOM or slow CPU offload.')

if not os.path.exists('introspection-mechanisms'):
    subprocess.run(['git','clone','https://github.com/safety-research/introspection-mechanisms.git'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers','accelerate','safetensors','matplotlib'], check=True)

assert os.environ.get('HF_TOKEN'), 'Set HF_TOKEN (env var on the pod) - Gemma is gated'
sys.path.insert(0,'introspection-mechanisms/src'); sys.path.insert(0,'introspection-mechanisms/experiments')
print('env ready; HF_HOME =', os.environ['HF_HOME'])

## 1. Config + candidate list

The four canonical arms (benign / valence / harm_adjacent / harmful) drive the **premise check**; the full candidate pool (arms + mined label words) drives the **cell screen**. `MINE_LABELS=True` folds in Gemma-Scope-2 transcoder-label vocabulary near L37 - available for this model only - to widen the search for the scarce off-diagonal cells.

In [ ]:
# === cell 2: config + candidates ===
import numpy as np, pandas as pd, re, json
from transformers import AutoTokenizer

MODEL          = 'google/gemma-3-27b-it'
LAYER_FRACTION = 0.6      # -> L37/62, the published operating point
N_PAIRS        = 120      # harmful/harmless pairs for d_refusal
N_BASELINE     = 100      # baseline words subtracted when building concept vectors
TOPK           = 5        # refusal-subspace dimension (cone robustness)

try:
    from vector_utils import get_baseline_words
    BASELINE_WORDS = get_baseline_words(N_BASELINE)
except Exception as e:
    print('get_baseline_words import failed, inline fallback:', e)
    BASELINE_WORDS = ('time year people way day man thing woman life child world school state family '
        'student group country problem hand part place case week company system program question work '
        'government number night point home water room mother area money story fact month lot right study '
        'book eye job word business issue side kind head house service friend father power hour game line '
        'end member law car city community name').split()[:N_BASELINE]

ARMS4 = ['benign','valence','harm_adjacent','harmful']
POOLS = {
 'benign':        'bread orchid apple hammer table cloud river pencil garden blanket lamp spoon window carpet mountain bicycle kettle pillow notebook umbrella ladder basket curtain teapot'.split(),
 'valence':       'sadness grief failure disgust loneliness boredom regret shame misery despair sorrow gloom anxiety dread humiliation guilt envy frustration disappointment heartbreak melancholy anguish'.split(),
 'harm_adjacent': 'police surgery medicine soldier ambulance prison vaccine hospital court firefighter military security patrol autopsy forensics quarantine arrest verdict trial guard sentry'.split(),
 'harmful':       'weapon poison assault explosive gun knife bomb murder torture grenade missile venom arson massacre stabbing shooting cyanide anthrax warfare terror kidnapping strangulation'.split(),
}
HARM_CATEGORY = {'benign':False,'valence':False,'harm_adjacent':False,'harmful':True}

MINE_LABELS  = True                       # transcoder labels exist for gemma-3-27b; widens the off-diagonal search
LABEL_LAYERS = [36, 37, 38]; LABEL_WIDTH='16k'; LABEL_L0='small'
HARM_LEX = set(('weapon gun knife bomb explosive grenade poison toxin venom cyanide anthrax assault murder '
    'kill torture attack violence abuse rape terror arson stab shoot wound massacre molest kidnap hostage '
    'missile warfare bloodshed mutilate strangle lynch sabotage detonate slaughter homicide manslaughter').split())
STOP = set('the a an and or of to in on for with about into over that this these those from your you it is are '
    'features feature related concept concepts words word text tokens token model layer neuron activation'.split())

def _mine(tok):
    from pathlib import Path
    from huggingface_hub import hf_hub_download
    found = {}
    for L in LABEL_LAYERS:
        data, fn = None, f'gemma_scope_2_27b_transcoder_all_layer{L}_{LABEL_WIDTH}_{LABEL_L0}_labels.json'
        try: data = json.loads(Path(hf_hub_download('google/gemma-scope-2-27b-it', f'feature_labels/{fn}')).read_text())
        except Exception as e: print(f'  layer {L}: no labels ({e})'); continue
        for lab in data.values():
            for w in re.findall(r'[a-z]{3,}', str(lab).lower()):
                if w in STOP or w in found: continue
                if len(tok(' '+w, add_special_tokens=False)['input_ids']) == 1: found[w] = str(lab)
    return found

rows, seen = [], set()
for cat in ARMS4:
    for w in POOLS[cat]:
        if w in seen: continue
        seen.add(w); rows.append(dict(concept=w, category=cat, harm=HARM_CATEGORY[cat]))
if MINE_LABELS:
    _t = AutoTokenizer.from_pretrained(MODEL)
    mined = _mine(_t)
    for w, lab in mined.items():
        if w in seen: continue
        seen.add(w); rows.append(dict(concept=w, category='mined', harm=any(k in (w+' '+lab).lower() for k in HARM_LEX)))
    print(f'  mined {len(mined)} single-token label words')
CAND_BASE = pd.DataFrame(rows)
print(f'{len(CAND_BASE)} candidates | harmful-tagged: {int(CAND_BASE.harm.sum())} | categories:',
      dict(CAND_BASE.category.value_counts()))

## 2. Load 27B, build `d_refusal` + concept vectors  *(long cell: ~54GB download first run, then a few minutes of forward passes)*

Keeps **all-layer** vectors for the four arms (for the premise-check layer sweep) and L37 `cos` for every candidate (for the cell screen) - one forward pass per concept feeds both.

In [ ]:
# === cell 3: load model + collect activations ===
import torch, gc
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM
import refusal_prompts

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, device_map={'': 0}, output_hidden_states=True).eval()
tcfg = getattr(model.config, 'text_config', model.config)   # gemma-3 nests the text stack
N_LAYERS = tcfg.num_hidden_layers
INJ = int(N_LAYERS * LAYER_FRACTION)
print(f'loaded {MODEL}: {N_LAYERS} layers -> injection L{INJ}')

@torch.no_grad()
def pooled(user_text):
    prompt = tok.apply_chat_template([{'role':'user','content':user_text}], tokenize=False, add_generation_prompt=True)
    enc = tok(prompt, return_tensors='pt', add_special_tokens=False).to(0)
    hs = model(**enc, output_hidden_states=True).hidden_states
    start = 1 if enc['input_ids'].shape[1] > 1 else 0
    last = torch.stack([h[0, -1] for h in hs]).float().cpu()          # [L+1, d]
    mx   = torch.stack([h[0, start:].amax(0) for h in hs]).float().cpu()
    return last, mx

# d_refusal at ALL layers (Arditi diff-of-means) + refusal subspace at INJ
H  = torch.stack([pooled(p)[0] for p in tqdm(refusal_prompts.HARMFUL_PROMPTS[:N_PAIRS],  desc='harmful ')])
Ml = torch.stack([pooled(p)[0] for p in tqdm(refusal_prompts.HARMLESS_PROMPTS[:N_PAIRS], desc='harmless')])
d_all = H.mean(0) - Ml.mean(0)
d_all = d_all / (d_all.norm(dim=-1, keepdim=True) + 1e-8)             # unit per layer [L+1, d]
Uk = torch.linalg.svd((H - Ml.mean(0))[:, INJ, :], full_matrices=False).Vh[:TOPK].T   # [d, TOPK]

# baseline mean at ALL layers
bl, bm = [], []
for w in tqdm(BASELINE_WORDS, desc='baseline'):
    l, m = pooled(f'Tell me about {w}'); bl.append(l); bm.append(m)
base_last, base_mx = torch.stack(bl).mean(0), torch.stack(bm).mean(0)  # [L+1, d]

# concept vectors: L37 cos for every candidate; all-layer vectors for the four arms
def cos_inj(v):    return float(((v/(v.norm()+1e-8)) * d_all[INJ]).sum())
arm_last = {a: [] for a in ARMS4}; arm_mx = {a: [] for a in ARMS4}
out = CAND_BASE.copy(); cl, cm, sp = [], [], []
for cat, w in tqdm(list(zip(out.category, out.concept)), desc='concept vecs'):
    l, m = pooled(f'Tell me about {w}')
    vl, vm = l - base_last, m - base_mx
    cl.append(cos_inj(vl[INJ])); cm.append(cos_inj(vm[INJ]))
    sp.append(float((vl[INJ] @ Uk @ Uk.T).norm() / (vl[INJ].norm() + 1e-8)))
    if cat in ARMS4:
        arm_last[cat].append(vl); arm_mx[cat].append(vm)
out['model']=MODEL.split('/')[-1]; out['inj_layer']=INJ; out['n_layers']=N_LAYERS
out['cos_last']=cl; out['cos_maxpool']=cm; out['subspace_frac']=sp
arm_last = {a: torch.stack(v) for a,v in arm_last.items() if v}        # [n, L+1, d]
arm_mx   = {a: torch.stack(v) for a,v in arm_mx.items() if v}

del model; gc.collect(); torch.cuda.empty_cache()
OUT = '/workspace/cell_screen_out'; os.makedirs(OUT, exist_ok=True)
print('activations built; model unloaded')

## 3. G3 premise check - per-arm cos across layers + verdict

The canonical geometry gate: is the **harmful** arm more refusal-aligned than **benign** at L37, stably across pooling protocols? Fills the pending 27B row of the findings log. Outputs are **per-arm aggregates** - safe to keep.

In [ ]:
# === cell 4: G3 premise check (per-arm aggregate + layer sweep + verdict) ===
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def cos_layers(v_all):                                   # [L+1, d] -> [L+1]
    vn = v_all / (v_all.norm(dim=-1, keepdim=True) + 1e-8)
    return (vn * d_all).sum(-1)

layers = list(range(N_LAYERS + 1)); summary = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, (pname, store) in zip(axes, [('last', arm_last), ('maxpool', arm_mx)]):
    for a in ARMS4:
        C = torch.stack([cos_layers(store[a][i]) for i in range(len(store[a]))])   # [n, L+1]
        m, se = C.mean(0), C.std(0)/(len(C)**0.5)
        ax.plot(layers, m, label=a); ax.fill_between(layers, m-se, m+se, alpha=0.15)
        summary[(pname, a)] = float(m[INJ])
    ax.axvline(INJ, ls='--', c='k', lw=0.8); ax.set_title(f'{pname}  (inj L{INJ})'); ax.set_xlabel('layer')
axes[0].set_ylabel('cos(v_concept, d_refusal)'); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/g3_layer_sweep_27b.png', dpi=110); plt.show()

print(f'per-arm cos at injection layer L{INJ}:')
for p in ['last', 'maxpool']:
    hb, bb = summary[(p,'harmful')], summary[(p,'benign')]
    va, ha = summary[(p,'valence')], summary[(p,'harm_adjacent')]
    print(f'  [{p:7s}] harmful={hb:+.3f} valence={va:+.3f} harm_adj={ha:+.3f} benign={bb:+.3f}'
          f'  | d(harm-benign)={hb-bb:+.3f}')
print('  subspace_frac (top-%d) by arm:' % TOPK,
      {a: round(float(out[out.category==a].subspace_frac.mean()), 3) for a in ARMS4})

holds  = summary[('last','harmful')] > summary[('last','benign')] and summary[('maxpool','harmful')] > summary[('maxpool','benign')]
stable = (summary[('last','harmful')] > summary[('last','benign')]) == (summary[('maxpool','harmful')] > summary[('maxpool','benign')])
print(f'\nVERDICT: premise {"HOLDS" if holds else "FAILS"}  (harmful>benign both protocols={holds}; ordering stable={stable})')

pd.DataFrame([{'protocol':p,'arm':a,'cos_inj':summary[(p,a)]} for p in ['last','maxpool'] for a in ARMS4]
             ).to_csv(f'{OUT}/g3_per_arm_aggregate_27b.csv', index=False)
print('saved ->', f'{OUT}/g3_per_arm_aggregate_27b.csv', '+ g3_layer_sweep_27b.png')

## 4. Four-cell screen - report + save

In [ ]:
# === cell 5: cell-screen 2x2 report + save (scalars + words only) ===
thr = float(out.cos_last.median())
rep = out.assign(geom=np.where(out.cos_last >= thr, 'close', 'distant'))
print(f'### {MODEL}  (inj L{INJ}/{N_LAYERS}, split cos_last={thr:+.3f})')
print(rep.pivot_table(index='harm', columns='geom', values='concept', aggfunc='count', fill_value=0))
for mask, title in [((rep.harm)&(rep.geom=='distant'), 'HARMFUL + DISTANT (empty => harm~cos collinear at L37 => dissociate via abliteration, not selection)'),
                    ((~rep.harm)&(rep.geom=='close'),  'HARMLESS + CLOSE  (valence/topic coupling to refusal)')]:
    sub = rep[mask].sort_values('cos_last')
    print(f'\n-- {title}  (n={len(sub)})')
    for _, r in sub.iterrows():
        print(f'   {r.concept:16s} cos_last={r.cos_last:+.3f} cos_max={r.cos_maxpool:+.3f} [{r.category}]')

path = f'{OUT}/cell_assignments_gemma3_27b.csv'
out.to_csv(path, index=False)     # never write vectors/activations here
print('\nsaved ->', path)

## 5. Get the files, then tear down

Three deliverables in `/workspace/cell_screen_out/`: `g3_per_arm_aggregate_27b.csv`, `g3_layer_sweep_27b.png`, `cell_assignments_gemma3_27b.csv`.

**Download:** in Jupyter's file browser, right-click each -> **Download**. (Or from your laptop: in the pod terminal `runpodctl send <file>`, then `runpodctl receive <code>` locally.)

**Then stop/terminate the pod.** The ~54GB model cache on `/workspace` is rebuildable, not a backup - don't keep a volume alive just to hold it (regenerate, never archive). The three files are the deliverables.

**Timing:** first run is dominated by the ~54GB download + load (~15-25 min); the ~600-700 forward passes are a few minutes. Expect **under an hour, ~\$1-2** on an A100 80GB.